In [ ]:
# Imports
import pandas as pd
import numpy as np
import torch as pt
import torch.nn as nn

In [ ]:
# Load and prep data

df = pd.read_csv('fhfa-hpi-data.csv')
df['Date'] = pd.to_datetime(df['Date']) # Instantiates the dates as Datetime objects
df = df.sort_values('Date').reset_index(drop=True)
df['HPI_YoY'] = df['Index'].pct_change(12) * 100
df = df.dropna()[['Date', 'HPI_YoY']]

In [ ]:
# Create X (current) and y (next month) pairs
X = df['HPI_YoY'].values[:-1].reshape(-1, 1) 
y = df['HPI_YoY'].values[1:].reshape(-1, 1)

In [ ]:
# Normalize with numpy (z-score)
X_mean, X_std = X.mean(), X.std()
X_scaled = (X - X_mean) / X_std

In [ ]:
# Train/test split (keep time order)
split_idx = int(len(X) * 0.8)
X_train, X_test = X_scaled[:split_idx], X_scaled[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

In [ ]:
# Convert to tensors
X_train = pt.FloatTensor(X_train)
X_test = pt.FloatTensor(X_test)
y_train = pt.FloatTensor(y_train)
y_test = pt.FloatTensor(y_test)

In [ ]:
# The Neural Network
class SimpleNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(1, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
    
    def forward(self, x):
        return self.layers(x)